# 06-B. legacy Event Context Ablation

이 노트북은 기존 LightGBM 모델에 추가한 과거 fault/task event context feature가 실제 holdout 일반화에 도움이 되는지 비교한다.

원칙:
- 메인 `lgbm_risk_scores.csv`를 덮어쓰지 않는다.
- 미래 fault까지 남은 시간은 feature로 사용하지 않는다.
- 모든 variant는 같은 row, 같은 split, 같은 threshold 선택 규칙으로 비교한다.
- 결과는 paper-aligned canonical이 아니라 legacy ablation archive로 보존한다.


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import average_precision_score, precision_recall_fscore_support, roc_auc_score

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 240)


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for path in [current, *current.parents]:
        if (path / 'pyproject.toml').exists():
            return path
    raise FileNotFoundError('pyproject.toml not found')


PROJECT_ROOT = find_project_root(Path.cwd())
FEATURE_DIR = PROJECT_ROOT / 'data' / 'processed' / 'ml_features'
BASELINE_DIR = PROJECT_ROOT / 'data' / 'processed' / 'ml_baseline'
ALIGNMENT_DIR = PROJECT_ROOT / 'data' / 'processed' / 'label_alignment'
RISK_DIR = PROJECT_ROOT / 'data' / 'processed' / 'ml_risk'

TRAINABLE_WINDOWS_PATH = FEATURE_DIR / 'trainable_windows.csv'
ML_WINDOW_DATASET_PATH = PROJECT_ROOT / 'data' / 'processed' / 'ml_windows' / 'ml_window_dataset.csv'
FEATURE_COLUMNS_PATH = FEATURE_DIR / 'feature_columns.csv'
ANOMALY_SCORES_PATH = BASELINE_DIR / 'anomaly_baseline_scores.csv'
FAULT_ALIGNMENT_PATH = ALIGNMENT_DIR / 'fault_alignment.csv'
DISTURBANCE_ALIGNMENT_PATH = ALIGNMENT_DIR / 'disturbance_alignment.csv'

ABLATION_PATH = RISK_DIR / 'lgbm_event_context_ablation.csv'
ABLATION_GROUP_PATH = RISK_DIR / 'lgbm_event_context_ablation_group_summary.csv'

PRIMARY_SPLIT_COLUMN = 'split_event_based'
RANDOM_STATE = 42
EVENT_CONTEXT_SENTINEL_DAYS = 9999.0
RECENT_EVENT_WINDOWS_DAYS = [30, 60, 90]

RISK_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
trainable_windows = pd.read_csv(TRAINABLE_WINDOWS_PATH)
ml_window_dataset = pd.read_csv(ML_WINDOW_DATASET_PATH)
feature_columns = pd.read_csv(FEATURE_COLUMNS_PATH)
anomaly_scores = pd.read_csv(ANOMALY_SCORES_PATH)
fault_alignment = pd.read_csv(FAULT_ALIGNMENT_PATH)
disturbance_alignment = pd.read_csv(DISTURBANCE_ALIGNMENT_PATH)

KEY_COLUMNS = ['manufacturer', 'substation_id', 'window_start', 'window_end']
TRAINING_CONTROL_COLUMNS = [
    column for column in [
        'use_for_supervised_training',
        'normal_reference_outlier',
        'normal_reference_outlier_count',
        'normal_reference_filter_reason',
    ]
    if column in ml_window_dataset.columns
]
if TRAINING_CONTROL_COLUMNS:
    trainable_windows = trainable_windows.drop(columns=[column for column in TRAINING_CONTROL_COLUMNS if column in trainable_windows.columns])
    trainable_windows = trainable_windows.merge(
        ml_window_dataset[KEY_COLUMNS + TRAINING_CONTROL_COLUMNS],
        on=KEY_COLUMNS,
        how='left',
        validate='one_to_one',
    )

ANOMALY_COLUMNS = KEY_COLUMNS + ['anomaly_score', 'anomaly_threshold', 'anomaly_label', 'iforest_anomaly_score']
available_anomaly_columns = [column for column in ANOMALY_COLUMNS if column in anomaly_scores.columns]
modeling_df = trainable_windows.merge(
    anomaly_scores[available_anomaly_columns],
    on=KEY_COLUMNS,
    how='left',
    validate='one_to_one',
)
if modeling_df['anomaly_score'].isna().any():
    raise ValueError('anomaly_score missing after merge')

if 'use_for_supervised_training' in modeling_df.columns:
    train_filter_mask = ~(modeling_df['split_time_based'].eq('train') & modeling_df['use_for_supervised_training'].eq(False))
    modeling_df = modeling_df.loc[train_filter_mask].copy()

modeling_df['risk_target'] = (modeling_df['label'] == 'pre_fault').astype(int)
for column in ['window_end', 'window_start']:
    modeling_df[column] = pd.to_datetime(modeling_df[column], errors='coerce')
for column in ['report_date', 'window_start', 'window_end']:
    if column in fault_alignment.columns:
        fault_alignment[column] = pd.to_datetime(fault_alignment[column], errors='coerce')
for column in ['event_start', 'window_start', 'window_end']:
    if column in disturbance_alignment.columns:
        disturbance_alignment[column] = pd.to_datetime(disturbance_alignment[column], errors='coerce')


def previous_event_features(windows: pd.DataFrame, events: pd.DataFrame, event_time_column: str, prefix: str) -> pd.DataFrame:
    result = pd.DataFrame(index=windows.index)
    days_column = f'days_since_last_{prefix}_event'
    has_column = f'has_previous_{prefix}_event'
    result[days_column] = EVENT_CONTEXT_SENTINEL_DAYS
    result[has_column] = 0
    for days in RECENT_EVENT_WINDOWS_DAYS:
        result[f'recent_{prefix}_{days}d'] = 0
    required_columns = {'manufacturer', 'substation_id', event_time_column}
    if events.empty or not required_columns.issubset(events.columns):
        return result
    clean_events = events.dropna(subset=[event_time_column]).copy()
    if clean_events.empty:
        return result
    event_groups = {
        key: group[event_time_column].sort_values().to_numpy(dtype='datetime64[ns]')
        for key, group in clean_events.groupby(['manufacturer', 'substation_id'], dropna=False)
    }
    for key, group_index in windows.groupby(['manufacturer', 'substation_id'], dropna=False).groups.items():
        event_times = event_groups.get(key)
        if event_times is None or len(event_times) == 0:
            continue
        window_times = windows.loc[group_index, 'window_start'].to_numpy(dtype='datetime64[ns]')
        positions = np.searchsorted(event_times, window_times, side='left') - 1
        valid = positions >= 0
        if not valid.any():
            continue
        target_index = np.asarray(list(group_index))[valid]
        deltas = (window_times[valid] - event_times[positions[valid]]) / np.timedelta64(1, 'D')
        deltas = np.maximum(deltas.astype(float), 0.0)
        result.loc[target_index, days_column] = deltas
        result.loc[target_index, has_column] = 1
        for days in RECENT_EVENT_WINDOWS_DAYS:
            result.loc[target_index, f'recent_{prefix}_{days}d'] = (deltas <= days).astype('int8')
    return result

usable_fault_events = fault_alignment.copy()
if 'is_usable' in usable_fault_events.columns:
    usable_fault_events = usable_fault_events[usable_fault_events['is_usable'].eq(True)].copy()
usable_fault_events = usable_fault_events.rename(columns={'report_date': 'event_time'})
usable_fault_events = usable_fault_events[['manufacturer', 'substation_id', 'event_time']].dropna(subset=['event_time'])

usable_disturbances = disturbance_alignment.copy()
if 'is_usable' in usable_disturbances.columns:
    usable_disturbances = usable_disturbances[usable_disturbances['is_usable'].eq(True)].copy()
usable_disturbances = usable_disturbances.rename(columns={'event_start': 'event_time'})
usable_disturbances = usable_disturbances[['manufacturer', 'substation_id', 'disturbance_type', 'event_time']].dropna(subset=['event_time'])
usable_task_events = usable_disturbances[~usable_disturbances['disturbance_type'].fillna('').str.lower().eq('fault')].copy()
usable_any_events = pd.concat(
    [
        usable_fault_events.assign(event_source='fault'),
        usable_task_events[['manufacturer', 'substation_id', 'event_time']].assign(event_source='task'),
    ],
    ignore_index=True,
)

for event_context in [
    previous_event_features(modeling_df, usable_fault_events, 'event_time', 'fault'),
    previous_event_features(modeling_df, usable_task_events, 'event_time', 'task'),
    previous_event_features(modeling_df, usable_any_events, 'event_time', 'any'),
]:
    for column in event_context.columns:
        modeling_df[column] = event_context[column]
modeling_df['post_fault_recent_90d'] = modeling_df['recent_fault_90d'].astype('int8')
modeling_df['post_task_recent_90d'] = modeling_df['recent_task_90d'].astype('int8')

fault_event_order = (
    modeling_df.dropna(subset=['fault_event_id'])
    .groupby('fault_event_id')['window_end']
    .max()
    .sort_values()
)
fault_event_ids = fault_event_order.index.to_list()
train_event_end = int(np.floor(len(fault_event_ids) * 0.70))
validation_event_end = int(np.floor(len(fault_event_ids) * 0.85))
fault_event_split_map = {}
for event_index, fault_event_id in enumerate(fault_event_ids):
    if event_index < train_event_end:
        fault_event_split_map[fault_event_id] = 'train'
    elif event_index < validation_event_end:
        fault_event_split_map[fault_event_id] = 'validation'
    else:
        fault_event_split_map[fault_event_id] = 'holdout'
modeling_df['split_event_based'] = modeling_df['split_time_based']
fault_event_mask = modeling_df['fault_event_id'].notna()
modeling_df.loc[fault_event_mask, 'split_event_based'] = modeling_df.loc[fault_event_mask, 'fault_event_id'].map(fault_event_split_map)

print(modeling_df.shape)
display(pd.crosstab(modeling_df['split_event_based'], modeling_df['label']))


(2555, 182)


label,normal,pre_fault
split_event_based,,
holdout,267,110
train,1262,472
validation,271,173


In [3]:
selected_feature_columns = (
    feature_columns.loc[feature_columns['selected_for_baseline'] == True, 'column_name']
    .dropna()
    .tolist()
)
CUMULATIVE_ABSOLUTE_FEATURES = {
    f'p_net_meter_{meter}__{stat}'
    for meter in ['energy', 'volume']
    for stat in ['first', 'last', 'min', 'max', 'mean']
}
guarded_feature_columns = [column for column in selected_feature_columns if column not in CUMULATIVE_ABSOLUTE_FEATURES]
base_context_columns = [column for column in ['anomaly_score', 'disturbance_count', 'maintenance_related'] if column in modeling_df.columns]

event_days_columns = [
    'days_since_last_fault_event',
    'days_since_last_task_event',
    'days_since_last_any_event',
]
event_fault_columns = [
    'days_since_last_fault_event',
    'has_previous_fault_event',
    'recent_fault_30d',
    'recent_fault_60d',
    'recent_fault_90d',
    'post_fault_recent_90d',
]
event_task_any_columns = [
    'days_since_last_task_event',
    'has_previous_task_event',
    'recent_task_30d',
    'recent_task_60d',
    'recent_task_90d',
    'post_task_recent_90d',
    'days_since_last_any_event',
    'has_previous_any_event',
    'recent_any_30d',
    'recent_any_60d',
    'recent_any_90d',
]
event_flag_columns = [column for column in [*event_fault_columns, *event_task_any_columns] if not column.startswith('days_since')]
event_full_columns = [
    'days_since_last_fault_event',
    'has_previous_fault_event',
    'recent_fault_30d',
    'recent_fault_60d',
    'recent_fault_90d',
    'days_since_last_task_event',
    'has_previous_task_event',
    'recent_task_30d',
    'recent_task_60d',
    'recent_task_90d',
    'days_since_last_any_event',
    'has_previous_any_event',
    'recent_any_30d',
    'recent_any_60d',
    'recent_any_90d',
    'post_fault_recent_90d',
    'post_task_recent_90d',
]

variant_extra_columns = {
    'guarded_no_event_context': [],
    'event_days_only': event_days_columns,
    'event_flags_only': event_flag_columns,
    'event_fault_only': event_fault_columns,
    'event_task_any_only': event_task_any_columns,
    'event_context_full': event_full_columns,
}

train_mask = modeling_df[PRIMARY_SPLIT_COLUMN].eq('train')
validation_mask = modeling_df[PRIMARY_SPLIT_COLUMN].eq('validation')
y_all = modeling_df['risk_target'].astype(int)
threshold_candidates = np.round(np.arange(0.05, 0.96, 0.05), 2)


def safe_roc_auc(y_true, y_score):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return float(roc_auc_score(y_true, y_score))


def safe_average_precision(y_true, y_score):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return float(average_precision_score(y_true, y_score))


def threshold_metrics(y_true, y_score, threshold):
    y_pred = (y_score >= threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', zero_division=0)
    negative_mask = y_true == 0
    positive_mask = y_true == 1
    fp = int(((y_pred == 1) & negative_mask).sum())
    fn = int(((y_pred == 0) & positive_mask).sum())
    tn = int(((y_pred == 0) & negative_mask).sum())
    tp = int(((y_pred == 1) & positive_mask).sum())
    return {
        'precision': float(precision),
        'recall': float(recall),
        'f1': float(f1),
        'false_positive_rate': float(fp / negative_mask.sum()) if negative_mask.sum() else np.nan,
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'tn': tn,
    }


def prepare_x(columns):
    X = modeling_df[columns].copy()
    for column in X.columns:
        if X[column].dtype == 'bool':
            X[column] = X[column].astype('int8')
        elif X[column].dtype == 'object':
            X[column] = pd.to_numeric(X[column], errors='coerce')
    if X.isna().any().any():
        missing = X.isna().sum()
        missing = missing[missing > 0].sort_values(ascending=False)
        raise ValueError(f'missing values in X: {missing.head(20)}')
    return X

metric_rows = []
group_rows = []
importance_rows = []
for variant_name, extra_columns in variant_extra_columns.items():
    feature_columns_for_variant = []
    for column in [*guarded_feature_columns, *base_context_columns, *extra_columns]:
        if column in modeling_df.columns and column not in feature_columns_for_variant:
            feature_columns_for_variant.append(column)
    X_all = prepare_x(feature_columns_for_variant)
    model = LGBMClassifier(
        objective='binary',
        n_estimators=150,
        learning_rate=0.04,
        num_leaves=15,
        min_child_samples=50,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.1,
        reg_lambda=1.0,
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1,
    )
    model.fit(
        X_all.loc[train_mask],
        y_all.loc[train_mask],
        eval_set=[(X_all.loc[validation_mask], y_all.loc[validation_mask])],
        eval_metric='average_precision',
    )
    y_score_all = model.predict_proba(X_all)[:, 1]
    validation_scores = y_score_all[validation_mask.to_numpy()]
    validation_true = y_all.loc[validation_mask].to_numpy()
    threshold_table = pd.DataFrame([
        {'threshold': threshold, **threshold_metrics(validation_true, validation_scores, threshold)}
        for threshold in threshold_candidates
    ]).sort_values(['f1', 'recall', 'precision'], ascending=[False, False, False]).reset_index(drop=True)
    selected_threshold = float(threshold_table.loc[0, 'threshold'])

    for split_name in ['train', 'validation', 'holdout']:
        split_mask = modeling_df[PRIMARY_SPLIT_COLUMN].eq(split_name).to_numpy()
        y_true = y_all.to_numpy()[split_mask]
        y_score = y_score_all[split_mask]
        row = {
            'variant': variant_name,
            'split': split_name,
            'row_count': int(split_mask.sum()),
            'normal_count': int((modeling_df.loc[split_mask, 'label'] == 'normal').sum()),
            'pre_fault_count': int((modeling_df.loc[split_mask, 'label'] == 'pre_fault').sum()),
            'feature_count': len(feature_columns_for_variant),
            'event_feature_count': len([column for column in extra_columns if column in feature_columns_for_variant]),
            'selected_high_threshold': selected_threshold,
            'roc_auc': safe_roc_auc(y_true, y_score),
            'average_precision': safe_average_precision(y_true, y_score),
            **threshold_metrics(y_true, y_score, selected_threshold),
        }
        metric_rows.append(row)

    scored = modeling_df.copy()
    scored['risk_probability_ablation'] = y_score_all
    scored['pred_high_ablation'] = (scored['risk_probability_ablation'] >= selected_threshold).astype(int)
    focus = scored[
        scored['manufacturer'].eq('manufacturer 2')
        & scored['configuration_type'].eq('SH')
        & scored[PRIMARY_SPLIT_COLUMN].eq('holdout')
    ]
    for (label, substation_id), group in focus.groupby(['label', 'substation_id'], dropna=False):
        group_rows.append(
            {
                'variant': variant_name,
                'label': label,
                'substation_id': substation_id,
                'rows': len(group),
                'risk_mean': float(group['risk_probability_ablation'].mean()),
                'risk_median': float(group['risk_probability_ablation'].median()),
                'high_or_critical_count': int(group['pred_high_ablation'].sum()),
                'selected_high_threshold': selected_threshold,
            }
        )

    importance = pd.DataFrame({'feature': feature_columns_for_variant, 'importance': model.feature_importances_})
    importance = importance.sort_values('importance', ascending=False).head(15).reset_index(drop=True)
    for rank, row in importance.iterrows():
        importance_rows.append({'variant': variant_name, 'rank': rank + 1, 'feature': row['feature'], 'importance': int(row['importance'])})

ablation_df = pd.DataFrame(metric_rows)
group_summary_df = pd.DataFrame(group_rows)
importance_summary_df = pd.DataFrame(importance_rows)
ablation_df.to_csv(ABLATION_PATH, index=False, encoding='utf-8-sig')
group_summary_df.to_csv(ABLATION_GROUP_PATH, index=False, encoding='utf-8-sig')
importance_summary_df.to_csv(RISK_DIR / 'lgbm_event_context_ablation_feature_importance.csv', index=False, encoding='utf-8-sig')

display(ablation_df[ablation_df['split'].eq('holdout')].sort_values('f1', ascending=False))
display(group_summary_df)
display(importance_summary_df.head(30))


,variant,split,row_count,normal_count,pre_fault_count,feature_count,event_feature_count,selected_high_threshold,roc_auc,average_precision,precision,recall,f1,false_positive_rate,tp,fp,fn,tn
8,event_flags_only,holdout,377,267,110,143,14,0.05,0.548791,0.306585,0.300000,0.954545,0.456522,0.917603,105,245,5,22
11,event_fault_only,holdout,377,267,110,135,6,0.10,0.500000,0.334473,0.278287,0.827273,0.416476,0.883895,91,236,19,31
14,event_task_any_only,holdout,377,267,110,140,11,0.75,0.544195,0.289828,0.323077,0.572727,0.413115,0.494382,63,132,47,135
17,event_context_full,holdout,377,267,110,146,17,0.70,0.562785,0.327083,0.307692,0.581818,0.402516,0.539326,64,144,46,123
5,event_days_only,holdout,377,267,110,132,3,0.80,0.576234,0.329687,0.313514,0.527273,0.393220,0.475655,58,127,52,140
2,guarded_no_event_context,holdout,377,267,110,129,0,0.30,0.446851,0.310641,0.253571,0.645455,0.364103,0.782772,71,209,39,58


,variant,label,substation_id,rows,risk_mean,risk_median,high_or_critical_count,selected_high_threshold
0,guarded_no_event_context,normal,11,28,0.909378,0.943624,28,0.30
1,guarded_no_event_context,normal,59,28,0.900790,0.910456,28,0.30
2,guarded_no_event_context,pre_fault,45,12,0.695960,0.747398,11,0.30
3,event_days_only,normal,11,28,0.960216,0.962234,28,0.80
4,event_days_only,normal,59,28,0.928741,0.930120,28,0.80
5,event_days_only,pre_fault,45,12,0.905936,0.930271,10,0.80
6,event_flags_only,normal,11,28,0.810731,0.814979,28,0.05
7,event_flags_only,normal,59,28,0.967190,0.971748,28,0.05
8,event_flags_only,pre_fault,45,12,0.867650,0.861653,12,0.05
9,event_fault_only,normal,11,28,0.892376,0.945616,28,0.10


,variant,rank,feature,importance
0,guarded_no_event_context,1,p_net_supply_temperature__max,89
1,guarded_no_event_context,2,s_hc1_supply_temperature__min,75
2,guarded_no_event_context,3,s_dhw_upper_storage_temperature__max,72
3,guarded_no_event_context,4,hc1_supply_temperature_gap__max_abs,69
4,guarded_no_event_context,5,p_net_meter_heat_power__min,60
5,guarded_no_event_context,6,anomaly_score,55
6,guarded_no_event_context,7,p_net_meter_heat_power__max,53
7,guarded_no_event_context,8,p_net_return_temperature__max,48
8,guarded_no_event_context,9,s_dhw_supply_temperature__mean,42
9,guarded_no_event_context,10,s_hc1_supply_temperature_setpoint__max,42
